# Notebook 002. Raster pre-processing

-------

Generates the analysis-ready rasters in `data/processed/rasters` from the raw data downloaded in notebook 001.

### ESA WorldCover composites — reference grid and EO layer

Builds the 10 m EPSG:3035 reference grid that every later layer aligns to, and the combined WorldCover earth-observation stack. The two AOI tiles (N45E024, N45E025) each contain four products (S2 RGB-NIR, NDVI percentiles, S1 VV/VH/ratio, SWIR). The ~20 m SWIR bands are resampled to 10 m. Each product is reprojected from its original EPSG:4326, mosaicked across tiles, stacked into 12 bands in a fixed order, and clipped to the AOI polygon. Output: float32, nodata −9999, band order recorded as descriptions.

In [ ]:
# Reference grid: WorldCover composites reprojected to EPSG:3035, mosaicked across
# the two AOI tiles, stacked, and clipped to the AOI polygon. Doubles as the
# WorldCover EO layer. 12 bands, float32, nodata -9999, 10 m.
import logging
import os
from pathlib import Path

import numpy as np
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.transform import from_origin

from utils import raster_io, terminology
from utils.paths import get_project_paths
from utils.vector_io import load_aoi

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

paths = get_project_paths()
NODATA = terminology.NODATA
RES = terminology.REF_RESOLUTION_M
N_THREADS = os.cpu_count()

WORLDCOVER_RAW = paths.raw / "rasters" / "worldcover_composites"
REF_GRID_PATH = paths.reference_grid

WC_TILES = ("N45E024", "N45E025")
WC_PRODUCTS = ("S2RGBNIR", "NDVI", "S1VVVHratio", "SWIR")


def wc_path(tile: str, product: str) -> Path:
    return WORLDCOVER_RAW / f"ESA_WorldCover_10m_2020_v100_{tile}_{product}.tif"


BAND_NAMES: tuple[str, ...] = (
    "s2_red",
    "s2_green",
    "s2_blue",
    "s2_nir",  # S2RGBNIR
    "ndvi_p90",
    "ndvi_p50",
    "ndvi_p10",  # NDVI percentiles
    "s1_vv",
    "s1_vh",
    "s1_vh_vv_ratio",  # S1 VV / VH / ratio
    "swir_b11",
    "swir_b12",  # SWIR (~20 m, upsampled)
)
PRODUCT_BAND_COUNTS = {"S2RGBNIR": 4, "NDVI": 3, "S1VVVHratio": 3, "SWIR": 2}

# Audit the raw inputs: source CRS, native resolution, band count, dtype, nodata.
print("[audit] raw WorldCover products (one per tile x product):")
for tile in WC_TILES:
    for product in WC_PRODUCTS:
        print(f"  {raster_io.audit_raster(wc_path(tile, product))}")

if REF_GRID_PATH.exists():
    print(f"[skip] reference grid already present: {REF_GRID_PATH.name}")
    grid = raster_io.open_reference_grid()
    print(f"[grid] {grid.width}x{grid.height} px, {RES:g} m, EPSG:{grid.crs.to_epsg()}")
else:
    aoi = load_aoi(dissolve=True)

    # Grid geometry from the AOI extent, snapped outward to a 10 m lattice.
    minx, miny, maxx, maxy = aoi.total_bounds
    minx = np.floor(minx / RES) * RES
    miny = np.floor(miny / RES) * RES
    maxx = np.ceil(maxx / RES) * RES
    maxy = np.ceil(maxy / RES) * RES
    width = int(round((maxx - minx) / RES))
    height = int(round((maxy - miny) / RES))
    grid = raster_io.ReferenceGrid(
        crs=CRS.from_user_input(terminology.CRS),
        transform=from_origin(minx, maxy, RES, RES),
        width=width,
        height=height,
    )
    print(
        f"[grid] derived from AOI extent: {width}x{height} px, {RES:g} m, "
        f"EPSG:{grid.crs.to_epsg()}, "
        f"bbox=({minx:.0f}, {miny:.0f}, {maxx:.0f}, {maxy:.0f})"
    )

    # Reproject each product's two tiles onto the grid (EPSG:4326 -> EPSG:3035,
    # ~10 m native to 10 m; SWIR ~20 m upsampled), then mosaic by coalescing valid
    # pixels across the tile seam. reproject_raster_to_ref reads each file's own
    # nodata (0 for S2/S1, 255 for NDVI/SWIR), mapping it to -9999 outside coverage.
    def mosaic_product(product: str) -> np.ndarray:
        out: np.ndarray | None = None
        for tile in WC_TILES:
            arr = raster_io.reproject_raster_to_ref(
                wc_path(tile, product),
                grid,
                resampling=Resampling.bilinear,
                dst_nodata=NODATA,
                dtype="float32",
                num_threads=N_THREADS,
            )
            if arr.shape[0] != PRODUCT_BAND_COUNTS[product]:
                raise ValueError(
                    f"{product} {tile}: {arr.shape[0]} bands, "
                    f"expected {PRODUCT_BAND_COUNTS[product]}."
                )
            out = arr if out is None else np.where(out == NODATA, arr, out)
        valid = int((out[0] != NODATA).sum())
        print(
            f"[mosaic] {product}: {len(WC_TILES)} tiles -> {out.shape[0]} band(s), "
            f"{valid:,} valid px in band 1"
        )
        return out

    stack = np.concatenate([mosaic_product(p) for p in WC_PRODUCTS], axis=0)
    if stack.shape[0] != len(BAND_NAMES):
        raise ValueError(f"{stack.shape[0]} bands but BAND_NAMES has {len(BAND_NAMES)}.")
    print(f"[stack] {stack.shape[0]} bands assembled in canonical order")

    # Clip to the AOI polygon: pixels outside the two Natura 2000 sites -> nodata.
    aoi_mask = raster_io.rasterize_mask(aoi.geometry, grid, all_touched=True)
    inside = int((aoi_mask != 0).sum())
    before = int((stack[0] != NODATA).sum())
    stack[:, aoi_mask == 0] = NODATA
    after = int((stack[0] != NODATA).sum())
    print(
        f"[clip] AOI polygon: band-1 valid {before:,} -> {after:,} px "
        f"(~{after / 100:,.0f} ha); mask interior {inside:,} px"
    )

    raster_io.write_geotiff(
        REF_GRID_PATH,
        stack,
        grid,
        dtype="float32",
        nodata=NODATA,
        band_descriptions=list(BAND_NAMES),
    )

# Verify: re-open (validates CRS and 10 m pixels) and audit with per-band stats.
grid = raster_io.open_reference_grid()
print(f"[verify] {raster_io.audit_raster(REF_GRID_PATH, with_stats=True)}")

### FABDEM — elevation, slope and heat-load index

Derives baseline topographic predictors (elevation, slope and heat load index, HLI) from the two FABDEM tiles. Slope and aspect are computed by Horn's method at FABDEM's native ~24 m resolution after reprojection to EPSG:3035; the heat-load index (McCune & Keon 2002, eq. 1; per-pixel, no neighbourhood window) is computed from slope and aspect. Elevation, slope and HLI are then resampled to the 10 m grid. Outputs are float32, nodata −9999, clipped to the AOI polygon.

In [ ]:
# FABDEM terrain: elevation, slope and heat-load index aligned to the reference grid.
# Terrain derivatives are computed at FABDEM's native resolution in EPSG:3035, then
# the continuous products are resampled to the 10 m grid; raw aspect, which wraps at
# 0/360 degrees, is never resampled. Outputs float32, nodata -9999, AOI-clipped.
import logging
import os

import numpy as np
import rasterio
from rasterio.crs import CRS
from rasterio.io import MemoryFile
from rasterio.merge import merge as rio_merge
from rasterio.warp import calculate_default_transform
from rasterio.warp import reproject as rio_reproject

from utils import predictors, raster_io, terminology
from utils.paths import get_project_paths
from utils.vector_io import load_aoi

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

paths = get_project_paths()
NODATA = terminology.NODATA
N_THREADS = os.cpu_count()

FABDEM_RAW = paths.raw / "rasters" / "fabdem"
FABDEM_OUT = paths.processed / "rasters" / "fabdem_10m"
FABDEM_OUT.mkdir(parents=True, exist_ok=True)

ELEV_PATH = FABDEM_OUT / "elevation_3035_10m.tif"
SLOPE_PATH = FABDEM_OUT / "slope_deg_3035_10m.tif"
HLI_PATH = FABDEM_OUT / "heat_load_index_3035_10m.tif"

print("[audit] raw FABDEM tiles:")
for tile in sorted(FABDEM_RAW.glob("*_FABDEM_V1-2.tif")):
    print(f"  {raster_io.audit_raster(tile)}")

if ELEV_PATH.exists() and SLOPE_PATH.exists() and HLI_PATH.exists():
    print("[skip] FABDEM terrain already present.")
else:
    grid = raster_io.open_reference_grid()
    aoi = load_aoi(dissolve=True)
    aoi_mask = raster_io.rasterize_mask(aoi.geometry, grid, all_touched=True)

    # Mosaic the native-CRS (EPSG:4326) tiles.
    tiles = sorted(FABDEM_RAW.glob("*_FABDEM_V1-2.tif"))
    srcs = [rasterio.open(t) for t in tiles]
    try:
        mosaic, mosaic_transform = rio_merge(srcs)
        src_crs = srcs[0].crs
        src_nodata = srcs[0].nodata
    finally:
        for s in srcs:
            s.close()
    print(
        f"[mosaic] {len(tiles)} FABDEM tiles -> {mosaic.shape[2]}x{mosaic.shape[1]} px, "
        f"{src_crs.to_string()}"
    )

    # Reproject the mosaic to EPSG:3035 at the source's native ground sampling, so
    # slope and aspect are differenced on a metric grid before any resampling.
    dst_crs = CRS.from_user_input(terminology.CRS)
    native_transform, native_w, native_h = calculate_default_transform(
        src_crs,
        dst_crs,
        mosaic.shape[2],
        mosaic.shape[1],
        *rasterio.transform.array_bounds(mosaic.shape[1], mosaic.shape[2], mosaic_transform),
    )
    elev_native = np.full((native_h, native_w), np.nan, dtype="float64")
    rio_reproject(
        source=mosaic[0],
        destination=elev_native,
        src_transform=mosaic_transform,
        src_crs=src_crs,
        src_nodata=src_nodata,
        dst_transform=native_transform,
        dst_crs=dst_crs,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear,
        num_threads=N_THREADS,
    )
    native_res_x = abs(native_transform.a)
    native_res_y = abs(native_transform.e)
    print(
        f"[reproject] mosaic {src_crs.to_string()} -> EPSG:{dst_crs.to_epsg()} at native "
        f"sampling: {native_w}x{native_h} px, {native_res_x:.2f}x{native_res_y:.2f} m"
    )

    # Derive slope, aspect and HLI on the native metric grid (degrees; HLI per
    # McCune & Keon 2002, equation 1).
    elev_for_terrain = np.where(np.isnan(elev_native), NODATA, elev_native).astype("float64")
    slope_native = predictors.horn_slope(elev_for_terrain, x_res=native_res_x, y_res=native_res_y)
    aspect_native = predictors.horn_aspect(elev_for_terrain, x_res=native_res_x, y_res=native_res_y)
    latitude_native = predictors.latitude_grid(native_transform, dst_crs, native_w, native_h)
    hli_native = predictors.heat_load_index(slope_native, aspect_native, latitude_native)
    sl = slope_native[slope_native != NODATA]
    hl = hli_native[hli_native != NODATA]
    print(
        f"[terrain] derived at {native_res_x:.2f} m: "
        f"slope {sl.min():.1f}-{sl.max():.1f} deg (mean {sl.mean():.1f}); "
        f"HLI {hl.min():.2f}-{hl.max():.2f} (mean {hl.mean():.2f})"
    )

    # Resample the continuous products (elevation, slope, HLI) up to the 10 m grid.
    # Aspect is intentionally not carried to 10 m; only HLI, derived from it, is.
    def to_reference_grid(array: np.ndarray) -> np.ndarray:
        profile = {
            "driver": "GTiff",
            "height": native_h,
            "width": native_w,
            "count": 1,
            "dtype": "float32",
            "crs": dst_crs,
            "transform": native_transform,
            "nodata": NODATA,
        }
        with MemoryFile() as mem:
            with mem.open(**profile) as tmp:
                tmp.write(array.astype("float32"), 1)
            return raster_io.reproject_raster_to_ref(
                mem.name,
                grid,
                resampling=Resampling.bilinear,
                src_nodata=NODATA,
                dst_nodata=NODATA,
                dtype="float32",
                num_threads=N_THREADS,
            )[0]

    elev_10m = to_reference_grid(elev_for_terrain)
    slope_10m = to_reference_grid(slope_native)
    hli_10m = to_reference_grid(hli_native)
    print(f"[resample] elevation, slope, HLI {native_res_x:.2f} m -> 10 m grid")

    # Clip each product to the AOI polygon and write.
    for array, path, name in (
        (elev_10m, ELEV_PATH, "elevation_m"),
        (slope_10m, SLOPE_PATH, "slope_deg"),
        (hli_10m, HLI_PATH, "heat_load_index"),
    ):
        before = int((array != NODATA).sum())
        array[aoi_mask == 0] = NODATA
        after = int((array != NODATA).sum())
        print(f"[clip] {name}: valid {before:,} -> {after:,} px (~{after / 100:,.0f} ha)")
        raster_io.write_geotiff(
            path, array, grid, dtype="float32", nodata=NODATA, band_descriptions=[name]
        )

print(f"[verify] {raster_io.audit_raster(ELEV_PATH, with_stats=True)}")
print(f"[verify] {raster_io.audit_raster(SLOPE_PATH, with_stats=True)}")
print(f"[verify] {raster_io.audit_raster(HLI_PATH, with_stats=True)}")

### OpenStreetMap — roads and distance-to-roads

Clips the Geofabrik Romania roads layer (`gis_osm_roads_free`) to the AOI with a 10 km buffer, classifies each segment into paved road, unpaved road or footpath by its OSM `fclass`, and writes both the classified roads (`osm_roads_aoi_buffer_10km.gpkg`) and a 3-band Euclidean distance-to-roads raster on the reference grid.

Classes are:
- Paved (motorway/trunk/primary/secondary/tertiary and links, residential, living_street, unclassified, road);
- Unpaved (service, track and track grades 1–5);
- Footpath (path, footway, steps, cycleway, bridleway, pedestrian, trail).
- Any other value (e.g. `busway`, `unknown`) is labelled "other" and excluded.

Outputs are float32, nodata −9999, clipped to the AOI polygon; band order is paved, unpaved, footpath.

In [ ]:
# OpenStreetMap roads: classify by fclass into paved / unpaved / footpath, store the
# network clipped to the AOI plus a 10 km buffer, and build a 3-band distance-to-roads
# raster. Distances are computed from the buffered network on a grid extended to the
# buffer's extent, then cropped to the reference grid and masked to the AOI polygon, so
# a road up to 10 km outside the AOI still sets the distance for boundary pixels.
# EPSG:3035 is an equal-area metric CRS, so distances are in true metres. Outputs are
# float32, nodata -9999; band order paved, unpaved, footpath.
import logging
import os

import geopandas
import numpy as np
from rasterio.transform import from_origin

from utils import raster_io, terminology, vector_io
from utils.paths import get_project_paths
from utils.vector_io import load_aoi, repair_geometries

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

paths = get_project_paths()
NODATA = terminology.NODATA
RES = terminology.REF_RESOLUTION_M

# Buffer placed around the AOI for both the stored road vector and the distance-transform
# domain. The distance bands are correct for any in-AOI pixel whose nearest road lies
# within this distance of the AOI; the previous paved band reached ~15.7 km, so raise
# this if the paved maximum below still moves between runs. The filename tracks this
# value so the two never drift.
ROADS_AOI_BUFFER_KM = 10
ROADS_AOI_BUFFER_M = ROADS_AOI_BUFFER_KM * 1_000.0

OSM_RAW = paths.raw / "vectors" / "open_street_map" / "romania.gpkg"
OSM_ROADS_LAYER = "gis_osm_roads_free"
ROADS_VEC_OUT = (
    paths.processed
    / "vectors"
    / "open_street_map"
    / f"osm_roads_aoi_buffer_{ROADS_AOI_BUFFER_KM}km.gpkg"
)
ROADS_RAST_OUT = (
    paths.processed / "rasters" / "roads_distance_10m" / "distance_to_roads_3035_10m.tif"
)
ROADS_VEC_OUT.parent.mkdir(parents=True, exist_ok=True)
ROADS_RAST_OUT.parent.mkdir(parents=True, exist_ok=True)

OSM_ROAD_GROUP_MAP = {
    "motorway": "paved_road",
    "trunk": "paved_road",
    "primary": "paved_road",
    "secondary": "paved_road",
    "tertiary": "paved_road",
    "unclassified": "paved_road",
    "residential": "paved_road",
    "living_street": "paved_road",
    "motorway_link": "paved_road",
    "trunk_link": "paved_road",
    "primary_link": "paved_road",
    "secondary_link": "paved_road",
    "tertiary_link": "paved_road",
    "road": "paved_road",
    "service": "unpaved_road",
    "track": "unpaved_road",
    "track_grade1": "unpaved_road",
    "track_grade2": "unpaved_road",
    "track_grade3": "unpaved_road",
    "track_grade4": "unpaved_road",
    "track_grade5": "unpaved_road",
    "path": "footpath",
    "footway": "footpath",
    "steps": "footpath",
    "cycleway": "footpath",
    "bridleway": "footpath",
    "pedestrian": "footpath",
    "trail": "footpath",
}
OSM_ROAD_GROUP_ORDER = ["paved_road", "unpaved_road", "footpath"]
LINE_TYPES = ("LineString", "MultiLineString")

grid = raster_io.open_reference_grid()
aoi = load_aoi(dissolve=True)  # EPSG:3035

# Distance-transform domain: the reference grid extended by the buffer on every side,
# so the buffered roads (which spill beyond the AOI bounding box) are all rasterised.
# Its extent equals the bounding box of the buffered AOI.
pad_px = int(round(ROADS_AOI_BUFFER_M / RES))
minx, miny, maxx, maxy = grid.bounds
padded = raster_io.ReferenceGrid(
    crs=grid.crs,
    transform=from_origin(minx - pad_px * RES, maxy + pad_px * RES, RES, RES),
    width=grid.width + 2 * pad_px,
    height=grid.height + 2 * pad_px,
)

if ROADS_VEC_OUT.exists() and ROADS_RAST_OUT.exists():
    print("[skip] OSM roads and distance raster already present.")
else:
    roads = geopandas.read_file(OSM_RAW, layer=OSM_ROADS_LAYER)
    source_crs = roads.crs
    print(f"[read] {OSM_ROADS_LAYER}: {len(roads):,} features, {source_crs.to_string()}")

    # Clip to the buffered AOI in the roads' own CRS (fast), then reproject the subset
    # to the project CRS. The buffer is formed in EPSG:3035 so its width is true metres.
    aoi_buf = aoi.geometry.buffer(ROADS_AOI_BUFFER_M)  # GeoSeries, EPSG:3035
    roads = geopandas.clip(roads, aoi_buf.to_crs(source_crs))
    roads = roads.to_crs(terminology.CRS)
    print(
        f"[buffer-clip] AOI + {ROADS_AOI_BUFFER_KM} km: {len(roads):,} features; "
        f"reprojected {source_crs.to_string()} -> {terminology.CRS}"
    )

    roads = repair_geometries(roads)
    roads = roads[roads.geom_type.isin(LINE_TYPES)].copy()

    # Classify and keep only the three road groups; this buffered network is both the
    # stored vector and the source for the distance bands.
    roads["fclass"] = roads["fclass"].astype("string").str.lower().str.strip()
    roads["road_group"] = roads["fclass"].map(OSM_ROAD_GROUP_MAP).fillna("other")
    roads_buf = roads.loc[
        roads["road_group"].isin(OSM_ROAD_GROUP_ORDER),
        ["osm_id", "fclass", "road_group", "geometry"],
    ].copy()

    summary = (
        roads_buf.assign(length_km=roads_buf.geometry.length / 1000.0)
        .groupby("road_group")
        .agg(features=("road_group", "size"), length_km=("length_km", "sum"))
        .round({"length_km": 2})
    )
    print("[classify] buffered network by group (written to vector):")
    print(summary.to_string())

    roads_buf.to_file(ROADS_VEC_OUT, driver="GPKG")
    print(
        f"[write] {ROADS_VEC_OUT.name}: {len(roads_buf):,} features "
        f"(AOI + {ROADS_AOI_BUFFER_KM} km)"
    )

    # Distance bands: transform on the extended grid from the buffered network, crop to
    # the reference grid, then mask to the AOI polygon.
    print(
        f"[grid+buffer] distance domain {padded.width}x{padded.height} px "
        f"(reference {grid.width}x{grid.height}, buffer {pad_px} px/side)"
    )
    aoi_mask = raster_io.rasterize_mask(aoi.geometry, grid, all_touched=True)
    bands = []
    for group in OSM_ROAD_GROUP_ORDER:
        lines = roads_buf.loc[roads_buf["road_group"] == group, "geometry"]
        if lines.empty:
            raise ValueError(f"No {group} features in the buffered AOI; cannot build its band.")
        dist_padded = raster_io.distance_to_lines_on_ref(lines, padded)
        dist = dist_padded[pad_px : pad_px + grid.height, pad_px : pad_px + grid.width].copy()
        dist[aoi_mask == 0] = NODATA
        inside = dist[dist != NODATA]
        print(
            f"[distance] {group}: {len(lines):,} lines (buffered) -> band, "
            f"max {inside.max():,.0f} m, {inside.size:,} valid px"
        )
        bands.append(dist)

    raster_io.write_geotiff(
        ROADS_RAST_OUT,
        np.stack(bands, axis=0),
        grid,
        dtype="float32",
        nodata=NODATA,
        band_descriptions=[f"dist_{g}_m" for g in OSM_ROAD_GROUP_ORDER],
    )

print(f"[verify] {vector_io.audit_vector(ROADS_VEC_OUT)}")
print(f"[verify] {raster_io.audit_raster(ROADS_RAST_OUT, with_stats=True)}")

## Copernicus HR-VPP

Aligns the High-Resolution Vegetation Phenology and Productivity (HR-VPP) Season 1 2020 parameters to the reference grid. The ten Vegetation Phenology and Productivity (VPP) parameters are each supplied as two Sentinel-2 tiles in different Universal Transverse Mercator (UTM) zones (T34TGR in EPSG:32634, T35TLL in EPSG:32635); each parameter is warped from its native 10 m UTM grid to EPSG:3035, mosaicked across the two tiles, clipped to the AOI polygon, and written as one band of a stacked raster.

The day-of-season layers are resampled by nearest neighbour to keep integer day-of-year and avoid interpolating across the no-season nodata; the continuous value, slope and productivity layers use bilinear. Output is float32, nodata −9999. Pixels where no vegetation season was detected (for example high bare rock) are nodata, so per-band valid counts may sit at or below the AOI pixel count.

In [ ]:
# Copernicus HR-VPP: align the Season 1 2020 phenology and productivity parameters to
# the reference grid. Each of the ten VPP parameters is supplied as two Sentinel-2 tiles
# in different UTM zones (T34TGR/EPSG:32634, T35TLL/EPSG:32635); each is warped to
# EPSG:3035, mosaicked across the two tiles, clipped to the AOI polygon, and written as
# one band of a stacked raster. Each parameter's own nodata is honoured. Day-of-season
# layers use nearest-neighbour resampling to keep integer day-of-year; the value, slope
# and productivity layers use bilinear. Band descriptions are the canonical vpp_* names
# from terminology. Output float32, nodata -9999.
import logging
import os

import numpy as np
from rasterio.enums import Resampling

from utils import raster_io, terminology
from utils.paths import get_project_paths
from utils.vector_io import load_aoi

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

paths = get_project_paths()
NODATA = terminology.NODATA
N_THREADS = os.cpu_count()

VPP_RAW = paths.raw / "rasters" / "copernicus_vpp"
VPP_OUT = paths.processed / "rasters" / "copernicus_vpp_10m"
VPP_OUT.mkdir(parents=True, exist_ok=True)
VPP_STACK_OUT = VPP_OUT / "copernicus_vpp_s1_2020_3035_10m.tif"

VPP_TILES = ("T34TGR", "T35TLL")  # UTM zones 34N and 35N
VPP_PARAMS = (
    "AMPL",
    "EOSD",
    "EOSV",
    "LSLOPE",
    "MAXV",
    "MINV",
    "RSLOPE",
    "SOSD",
    "SOSV",
    "SPROD",
)

# Canonical band descriptions come from terminology, the single source of truth.
# Assert the local parameter order matches it, so the file is written with the
# vpp_* names and the cell fails loudly if the two ever diverge.
VPP_BAND_NAMES = terminology._HRVPP_BANDS
assert tuple(f"vpp_{p.lower()}" for p in VPP_PARAMS) == VPP_BAND_NAMES, (
    "VPP_PARAMS order does not match terminology._HRVPP_BANDS; "
    "reconcile before writing band descriptions."
)

# Day-of-season layers are integer day-of-year, with 0 meaning no season detected;
# nearest neighbour preserves exact dates and does not interpolate across that nodata.
# The remaining continuous layers use bilinear.
VPP_RESAMPLING = {p: Resampling.bilinear for p in VPP_PARAMS}
VPP_RESAMPLING["SOSD"] = Resampling.nearest
VPP_RESAMPLING["EOSD"] = Resampling.nearest


def vpp_path(tile: str, param: str):
    return VPP_RAW / f"VPP_2020_S2_{tile}-010m_V101_s1_{param}.tif"


grid = raster_io.open_reference_grid()

if VPP_STACK_OUT.exists():
    print(f"[skip] HR-VPP stack already present: {VPP_STACK_OUT.name}")
else:
    aoi = load_aoi(dissolve=True)
    aoi_mask = raster_io.rasterize_mask(aoi.geometry, grid, all_touched=True)

    # Warp each parameter's two UTM tiles onto the grid and mosaic by coalescing valid
    # pixels. reproject_raster_to_ref reads each file's own nodata, so the three nodata
    # regimes (0 / 65535 / -32768) are honoured without a hardcoded map.
    def mosaic_param(param: str) -> np.ndarray:
        src_nd = raster_io.audit_raster(vpp_path(VPP_TILES[0], param)).nodata[0]
        out: np.ndarray | None = None
        for tile in VPP_TILES:
            arr = raster_io.reproject_raster_to_ref(
                vpp_path(tile, param),
                grid,
                resampling=VPP_RESAMPLING[param],
                dst_nodata=NODATA,
                dtype="float32",
                num_threads=N_THREADS,
            )[0]
            out = arr if out is None else np.where(out == NODATA, arr, out)
        out[aoi_mask == 0] = NODATA
        valid = int((out != NODATA).sum())
        print(
            f"[mosaic] {param}: src nodata={src_nd:g}, "
            f"resampling={VPP_RESAMPLING[param].name}, "
            f"2 UTM tiles -> {valid:,} valid px in AOI"
        )
        return out

    stack = np.stack([mosaic_param(p) for p in VPP_PARAMS], axis=0)
    print(f"[stack] {stack.shape[0]} VPP parameters assembled (EPSG:32634/32635 -> EPSG:3035)")

    raster_io.write_geotiff(
        VPP_STACK_OUT,
        stack,
        grid,
        dtype="float32",
        nodata=NODATA,
        band_descriptions=list(VPP_BAND_NAMES),
    )

print(f"[verify] {raster_io.audit_raster(VPP_STACK_OUT, with_stats=True)}")

## AlphaEarth

Aligns the AlphaEarth Foundations 2020 annual embedding to the reference grid. The embedding is downloaded in its native EPSG:32635 (UTM zone 35N) at 10 m with a 10 km buffer around the AOI, carrying 64 embedding dimensions in float32. It is reprojected once to EPSG:3035 on the reference grid, clipped to the AOI polygon, and written as a 64-band stack. The source declares no nodata, so any NaN cells produced by the source or by warping outside the embedding footprint are converted to -9999 before clipping. Bilinear resampling is used for the continuous embedding dimensions. Output is float32, nodata -9999.

In [ ]:
# AlphaEarth: align the 64-band annual embedding (2020) to the reference grid.
# The download is in native EPSG:32635 at 10 m with a 10 km AOI buffer, so it is
# reprojected once to EPSG:3035 on the reference grid and clipped to the AOI
# polygon. The source declares no nodata, so NaN cells are converted to -9999
# after the warp and before clipping. Continuous embedding dimensions use
# bilinear. Output float32, nodata -9999.
import logging
import os

import numpy as np
from rasterio.enums import Resampling

from utils import raster_io, terminology
from utils.paths import get_project_paths
from utils.vector_io import load_aoi

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

paths = get_project_paths()
NODATA = terminology.NODATA
N_THREADS = os.cpu_count()

ALPHAEARTH_RAW = paths.raw / "rasters" / "alphaearth" / "alphaearth_2020_aoi_32635.tif"
ALPHAEARTH_OUT = paths.processed / "rasters" / "alphaearth_10m"
ALPHAEARTH_OUT.mkdir(parents=True, exist_ok=True)
ALPHAEARTH_STACK_OUT = ALPHAEARTH_OUT / "alphaearth_2020_3035_10m.tif"

grid = raster_io.open_reference_grid()
print(f"[audit] {raster_io.audit_raster(ALPHAEARTH_RAW)}")

if ALPHAEARTH_STACK_OUT.exists():
    print(f"[skip] AlphaEarth stack already present: {ALPHAEARTH_STACK_OUT.name}")
else:
    aoi = load_aoi(dissolve=True)
    aoi_mask = raster_io.rasterize_mask(aoi.geometry, grid, all_touched=True)

    stack = raster_io.reproject_raster_to_ref(
        ALPHAEARTH_RAW,
        grid,
        resampling=Resampling.bilinear,
        dst_nodata=NODATA,
        dtype="float32",
        num_threads=N_THREADS,
    )
    print(
        f"[warp] 64 bands EPSG:32635 -> EPSG:3035 onto reference grid "
        f"({stack.shape[1]}x{stack.shape[2]} px)"
    )

    # The source declares no nodata, so NaN cells survived the warp; convert
    # them to the project sentinel so the file matches the project convention.
    n_nan = int(np.isnan(stack).sum())
    stack[np.isnan(stack)] = NODATA
    print(f"[nan-fill] {n_nan:,} NaN cells -> {NODATA:g}")

    stack[:, aoi_mask == 0] = NODATA
    valid = int((stack[0] != NODATA).sum())
    print(f"[clip] AOI polygon: {valid:,} valid px in band 1 (~{valid / 100:,.0f} ha)")

    raster_io.write_geotiff(
        ALPHAEARTH_STACK_OUT,
        stack,
        grid,
        dtype="float32",
        nodata=NODATA,
        band_descriptions=[f"alphaearth_a{i:02d}" for i in range(64)],
    )

print(f"[verify] {raster_io.audit_raster(ALPHAEARTH_STACK_OUT)}")  # metadata-only

## TESSERA

Aligns the TESSERA 2020 annual embedding to the reference grid. The embedding ships as 55 tiles on a 0.1° lat × 0.1° lon grid in EPSG:32635 at 10 m, each carrying 128 embedding dimensions in float32. Tiles are mosaicked in their native CRS, warped once to EPSG:3035, clipped to the AOI polygon, and written as a 128-band stack. The source declares no nodata, so any NaN cells produced by the source or by warping outside the mosaic footprint are converted to −9999 before clipping. Bilinear resampling is used for the continuous embedding dimensions. Output is float32, nodata −9999.

In [ ]:
# TESSERA: align the 128-band annual embedding (2020) to the reference grid. The
# 55 tiles share EPSG:32635 and 10 m resolution but not one pixel lattice (their
# origins differ by sub-pixel offsets), so each tile is warped straight onto the
# reference grid with bilinear resampling, later tiles overwriting earlier ones
# where neighbours overlap, and the result is clipped to the AOI polygon. Tiles
# are read whole, one at a time: GDAL decodes these pixel-interleaved tiles once
# per band when several are read through one virtual raster, which turns a
# minute of work into hours. The source declares no nodata, so NaN cells are
# converted to -9999 after the warp and before clipping. Output float32, nodata
# -9999.
import logging
import math
import os

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds

from utils import raster_io, terminology
from utils.paths import get_project_paths
from utils.vector_io import load_aoi

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)

paths = get_project_paths()
NODATA = terminology.NODATA
N_THREADS = os.cpu_count()

TESSERA_RAW = paths.raw / "rasters" / "tessera"
TESSERA_OUT = paths.processed / "rasters" / "tessera_10m"
TESSERA_OUT.mkdir(parents=True, exist_ok=True)
TESSERA_STACK_OUT = TESSERA_OUT / "tessera_2020_3035_10m.tif"

grid = raster_io.open_reference_grid()
tiles = sorted(p for ext in ("*.tif", "*.tiff") for p in TESSERA_RAW.rglob(ext))
print(f"[audit] {len(tiles)} TESSERA tiles in {TESSERA_RAW.name}")
print(f"  {raster_io.audit_raster(tiles[0])}  (representative tile)")

if TESSERA_STACK_OUT.exists():
    print(f"[skip] TESSERA stack already present: {TESSERA_STACK_OUT.name}")
else:
    aoi = load_aoi(dissolve=True)
    aoi_mask = raster_io.rasterize_mask(aoi.geometry, grid, all_touched=True)

    stack = np.full((128, grid.height, grid.width), NODATA, dtype="float32")
    for tile in tiles:
        with rasterio.open(tile) as src:
            source = src.read()
            src_crs, src_transform = src.crs, src.transform
            bounds = transform_bounds(src_crs, grid.crs, *src.bounds)
        # The tile's footprint on the reference grid, snapped outwards.
        col0 = max(math.floor((bounds[0] - grid.transform.c) / grid.resolution[0]), 0)
        row0 = max(math.floor((grid.transform.f - bounds[3]) / grid.resolution[1]), 0)
        col1 = min(math.ceil((bounds[2] - grid.transform.c) / grid.resolution[0]), grid.width)
        row1 = min(math.ceil((grid.transform.f - bounds[1]) / grid.resolution[1]), grid.height)
        if col1 <= col0 or row1 <= row0:
            continue
        window = np.full((128, row1 - row0, col1 - col0), NODATA, dtype="float32")
        reproject(
            source=source,
            destination=window,
            src_transform=src_transform,
            src_crs=src_crs,
            src_nodata=None,
            dst_transform=grid.transform * grid.transform.translation(col0, row0),
            dst_crs=grid.crs,
            dst_nodata=NODATA,
            resampling=Resampling.bilinear,
            num_threads=N_THREADS,
        )
        covered = window[0] != NODATA
        stack[:, row0:row1, col0:col1][:, covered] = window[:, covered]
    print(
        f"[warp] {len(tiles)} tiles, 128 bands EPSG:32635 -> EPSG:3035 onto reference grid "
        f"({stack.shape[1]}x{stack.shape[2]} px)"
    )

    # The source declares no nodata, so NaN cells survived the warp; convert
    # them to the project sentinel so the file matches the project convention.
    n_nan = int(np.isnan(stack).sum())
    stack[np.isnan(stack)] = NODATA
    print(f"[nan-fill] {n_nan:,} NaN cells -> {NODATA:g}")

    stack[:, aoi_mask == 0] = NODATA
    valid = int((stack[0] != NODATA).sum())
    print(f"[clip] AOI polygon: {valid:,} valid px in band 1 (~{valid / 100:,.0f} ha)")

    raster_io.write_geotiff(
        TESSERA_STACK_OUT,
        stack,
        grid,
        dtype="float32",
        nodata=NODATA,
        band_descriptions=[f"tessera_t{i:03d}" for i in range(128)],
    )

print(f"[verify] {raster_io.audit_raster(TESSERA_STACK_OUT)}")  # metadata-only

## European Forest Disturbance Atlas - disturbance mask

Builds a binary 1985-2020 forest-disturbance mask on the reference grid, used in
notebook 005 to remove disturbed areas from old-growth parcels. The atlas (Viana-Soto
and Senf, 2024) is a 39-band annual stack for 1985-2023 of 0/1 disturbance flags at
30 m in EPSG:3035; bands 1-36 are the years 1985-2020 screened by the study. The bands
are combined over the AOI window into a single ever-disturbed mask at native 30 m,
resampled to the 10 m grid by nearest neighbour (a categorical mask must not be
averaged), and clipped to the AOI. Output is uint8: 1 disturbed, 0 undisturbed, 255
nodata outside the AOI.

In [ ]:
# European Forest Disturbance Atlas: a binary "disturbed at any point 1985-2020" mask
# on the reference grid. The atlas is a 39-band annual stack (1985-2023) of 0/1 flags
# at 30 m in EPSG:3035; bands 1-36 cover 1985-2020. The bands are OR-ed over the AOI
# window at native 30 m, resampled to the 10 m grid by nearest neighbour, and clipped
# to the AOI. Output uint8: 1 disturbed, 0 undisturbed, 255 nodata outside the AOI.
import os

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.windows import from_bounds

from utils import raster_io
from utils.paths import get_project_paths
from utils.vector_io import load_aoi

paths = get_project_paths()
N_THREADS = os.cpu_count()

EFDA_RAW = (
    paths.raw
    / "rasters"
    / "european_forest_disturbance_atlas"
    / "annual_disturbances_1985_2023_romania.tif"
)
EFDA_OUT = paths.processed / "rasters" / "european_forest_disturbance_atlas_10m"
EFDA_OUT.mkdir(parents=True, exist_ok=True)
DISTURBANCE_OUT = EFDA_OUT / "disturbance_1985_2020_3035_10m.tif"

FIRST_YEAR = 1985  # atlas band 1 (confirm ascending order in the Zenodo readme)
LAST_YEAR_INCLUSIVE = 2020  # the manuscript screens disturbance 1985-2020
N_BANDS_USED = LAST_YEAR_INCLUSIVE - FIRST_YEAR + 1  # 36
MASK_NODATA = 255

grid = raster_io.open_reference_grid()

if DISTURBANCE_OUT.exists():
    print(f"[skip] disturbance mask already present: {DISTURBANCE_OUT.name}")
else:
    aoi = load_aoi(dissolve=True)  # EPSG:3035, the same CRS as the atlas

    with rasterio.open(EFDA_RAW) as src:
        if src.crs is None or src.crs.to_epsg() != 3035:
            raise ValueError(f"EFDA expected EPSG:3035, found {src.crs}.")
        # Window to the AOI bounds (same CRS) so the Romania-wide 39-band stack is
        # never read in full; pad outwards to integer pixels to cover the AOI edge.
        window = from_bounds(*aoi.total_bounds, transform=src.transform)
        window = window.round_offsets(op="floor").round_lengths(op="ceil")
        annual = src.read(list(range(1, N_BANDS_USED + 1)), window=window)  # (36, h, w)
        win_transform = src.window_transform(window)
        src_crs = src.crs

    # Disturbed in any year 1985-2020; 0 and nodata (255) both contribute 0.
    union_30m = (annual == 1).any(axis=0).astype("uint8")
    n30 = int(union_30m.sum())
    print(
        f"[build] EFDA bands 1-{N_BANDS_USED} ({FIRST_YEAR}-{LAST_YEAR_INCLUSIVE}) over the AOI "
        f"window {union_30m.shape[1]}x{union_30m.shape[0]} px at 30 m: {n30:,} disturbed "
        f"(~{n30 * 0.09:,.0f} ha)"
    )

    # Resample the binary mask 30 m -> 10 m by nearest neighbour (no class averaging).
    mask_10m = np.zeros(grid.shape, dtype="uint8")
    reproject(
        source=union_30m,
        destination=mask_10m,
        src_transform=win_transform,
        src_crs=src_crs,
        dst_transform=grid.transform,
        dst_crs=grid.crs,
        resampling=Resampling.nearest,
        num_threads=N_THREADS,
    )
    print("[resample] disturbance mask EPSG:3035 30 m -> EPSG:3035 10 m, nearest neighbour")

    # Confine to the AOI: outside the boundary is nodata.
    aoi_mask = raster_io.rasterize_mask(aoi.geometry, grid, all_touched=True)
    mask_10m[aoi_mask == 0] = MASK_NODATA

    n_dist = int((mask_10m == 1).sum())
    n_valid = int((mask_10m != MASK_NODATA).sum())
    print(
        f"[clip] disturbance mask on the 10 m grid: {n_dist:,} disturbed px "
        f"(~{n_dist / 100:,.0f} ha), {100 * n_dist / n_valid:.1f}% of {n_valid:,} AOI px"
    )

    raster_io.write_geotiff(
        DISTURBANCE_OUT,
        mask_10m[np.newaxis, :, :],
        grid,
        dtype="uint8",
        nodata=MASK_NODATA,
        band_descriptions=["disturbed_1985_2020"],
    )
    print(f"[write] {DISTURBANCE_OUT.name}")

print(f"[verify] {raster_io.audit_raster(DISTURBANCE_OUT, with_stats=True)}")